In [4]:
import sys
import os
from pathlib import Path
from src.utils import normalize_columns

root_path = str(Path(os.getcwd()).parent)
if root_path not in sys.path:
    sys.path.append(root_path)


ModuleNotFoundError: No module named 'app'

In [7]:
import os
import sys
import hashlib
import json
import re
import subprocess
import numpy as np
import pandas as pd
import fitz  # PyMuPDF
import spacy
import faiss
import onnxruntime as ort
from datetime import datetime
from pathlib import Path
from bs4 import BeautifulSoup
from transformers import AutoTokenizer

# --- 1. CONFIGURAÇÃO DE AMBIENTE E PATHS ---
BASE_DIR = Path(os.getcwd()).resolve()
MODEL_PATH = (BASE_DIR / "../app/resources/models/bertimbau_onnx").resolve()
INDEX_PATH = (BASE_DIR / "../app/resources/models/cbo_index.faiss").resolve()
CBO_CSV_PATH = (BASE_DIR / "../app/resources/data/cbo_base.csv").resolve()

# --- 2. INICIALIZAÇÃO AUTOMÁTICA DE MODELOS (RESILIÊNCIA) ---

def inicializar_spacy(modelo="pt_core_news_lg"):
    try:
        return spacy.load(modelo)
    except OSError:
        print(f"--> Modelo {modelo} não encontrado. Baixando...")
        subprocess.run([sys.executable, "-m", "spacy", "download", modelo])
        return spacy.load(modelo)

def inicializar_tokenizer():
    if not (MODEL_PATH / "vocab.txt").exists():
        print("--> Vocabulário não encontrado localmente. Baixando do Hub...")
        tk = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
        tk.save_pretrained(MODEL_PATH)
        return tk
    return AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)

# Inicialização Global
nlp = inicializar_spacy()
tokenizer = inicializar_tokenizer()
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
session = ort.InferenceSession(str(MODEL_PATH / "model.onnx"), sess_options, providers=['CPUExecutionProvider'])
model_inputs = [i.name for i in session.get_inputs()]

# --- 3. CORE DE EXTRAÇÃO E CONVERSÃO ---

def converter_para_texto_completo(caminho_arquivo):
    extensao = os.path.splitext(caminho_arquivo)[1].lower()
    try:
        if extensao == ".html":
            with open(caminho_arquivo, "r", encoding="utf-8", errors="ignore") as f:
                return BeautifulSoup(f.read(), "html.parser").get_text(separator=" ")
        elif extensao == ".pdf":
            doc = fitz.open(caminho_arquivo)
            return " ".join([pagina.get_text() for pagina in doc])
        elif extensao == ".txt":
            with open(caminho_arquivo, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
    except Exception as e:
        print(f"Erro ao converter {caminho_arquivo}: {e}")
    return ""

def extrair_metadados_iniciais(caminho_arquivo, texto_amostra):
    with open(caminho_arquivo, "rb") as f:
        binario = f.read()

    return {
        "id_arquivo": hashlib.sha256(binario).hexdigest(),
        "tamanho_documento": len(binario),
        "data_processamento": datetime.now().isoformat(),
        "poder_republica": "Judiciário" if any(x in texto_amostra for x in ["Tribunal", "TRT"]) else "Executivo",
        "orgao": re.search(r"(Ministério\sde\s[A-Z][a-zà-ù]+|Superintendência\sRegional)", texto_amostra).group(0) if re.search(r"(Ministério\sde\s[A-Z][a-zà-ù]+|Superintendência\sRegional)", texto_amostra) else "MTE",
        "cidade": re.search(r"([A-Z][a-zà-ù]+(?:\s[A-Z][a-zà-ù]+)*)\s?/\s?[A-Z]{2}", texto_amostra).group(1) if re.search(r"([A-Z][a-zà-ù]+(?:\s[A-Z][a-zà-ù]+)*)\s?/\s?[A-Z]{2}", texto_amostra) else "Não Identificada",
        "data_legal": re.search(r"(\d{2}/\d{2}/\d{4})", texto_amostra).group(0) if re.search(r"(\d{2}/\d{2}/\d{4})", texto_amostra) else None
    }

# --- 4. INTELIGÊNCIA ARTIFICIAL (NER + EMBEDDINGS) ---

def localizar_entidades_curso(texto):
    doc = nlp(texto[:100000]) # Limite para evitar estouro de memória
    cursos = []
    for chunk in doc.noun_chunks:
        if any(kw in chunk.text.lower() for kw in ["curso", "treinamento", "capacitação", "técnico"]):
            cursos.append(chunk.text.strip())
    return list(set(cursos))

def get_embeddings(texts, batch_size=16):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="np", padding=True, truncation=True, max_length=512)
        input_feed = {name: inputs[name] for name in model_inputs if name in inputs}
        outputs = session.run(None, input_feed)

        token_embeddings = outputs[0]
        mask = inputs["attention_mask"][:, :, np.newaxis]
        batch_mean = np.sum(token_embeddings * mask, axis=1) / np.maximum(np.sum(mask, axis=1), 1e-9)
        all_embeddings.append(batch_mean)
    return np.vstack(all_embeddings).astype('float32')

# --- 5. FLUXO PRINCIPAL ---

def processar_documento(caminho_arquivo, index_faiss, df_cbo):
    texto = converter_para_texto_completo(caminho_arquivo)
    if not texto: return None

    meta = extrair_metadados_iniciais(caminho_arquivo, texto[:2000])
    cursos = localizar_entidades_curso(texto)

    resultados_cbo = []
    if cursos:
        vetores = get_embeddings(cursos)
        faiss.normalize_L2(vetores)
        distancias, indices = index_faiss.search(vetores, k=1)

        for i, idx_cbo in enumerate(indices):
            resultados_cbo.append({
                "curso_detectado": cursos[i],
                "cbo_atribuida": df_cbo.iloc[idx_cbo[0]]['ocupacao'],
                "codigo_cbo": str(df_cbo.iloc[idx_cbo[0]]['codigo']),
                "confianca": float(distancias[i][0])
            })

    return {"metadados": meta, "analise_cbo": resultados_cbo}

# --- 6. PERSISTÊNCIA E EXECUÇÃO ---

def salvar_json(resultado):
    os.makedirs("../data/processed", exist_ok=True)
    with open(f"../data/processed/{resultado['metadados']['id_arquivo']}.json", "w", encoding="utf-8") as f:
        json.dump(resultado, f, indent=4, ensure_ascii=False)

def main():
    print("--> Carregando Recursos...")
    idx = faiss.read_index(str(INDEX_PATH))
    df_cbo = pd.read_csv(str(CBO_CSV_PATH))

    # Exemplo: Processar pasta raw
    pasta_entrada = "../data/raw/sei_v1"
    arquivos = [os.path.join(pasta_entrada, f) for f in os.listdir(pasta_entrada) if f.endswith(('.pdf', '.html'))]

    print(f"--> Processando {len(arquivos)} arquivos...")
    for arq in arquivos:
        res = processar_documento(arq, idx, df_cbo)
        if res:
            salvar_json(res)
            print(f"✅ OK: {os.path.basename(arq)}")

if __name__ == "__main__":
    main()

--> Vocabulário não encontrado localmente. Baixando do Hub...


NoSuchFile: [ONNXRuntimeError] : 3 : NO_SUCHFILE : Load model from /home/gabrielsousa/workspace-mte/processador_documentos/processador-documentos-backend/app/resources/models/bertimbau_onnx/model.onnx failed:Load model /home/gabrielsousa/workspace-mte/processador_documentos/processador-documentos-backend/app/resources/models/bertimbau_onnx/model.onnx failed. File doesn't exist